# Single-Compartment Neurons

`braincell` builds every neuron on the Hodgkin–Huxley (HH) framework, and every neuron is a {class}`braincell.Cell`:

- A **point neuron** has a single isopotential compartment. `braincell.Cell.point(...)` makes one; it is short for `braincell.Cell(braincell.Morphology.from_soma(...), cv_policy=braincell.Lumped())`. It suits neurons whose spatial structure can be ignored.
- A **morphological neuron** is a `Cell` on a full {class}`~braincell.Morphology`, divided into control volumes — covered in the [Cell](cell.ipynb) tutorial.

Both are declared the same way: `paint` puts ion species and channels on the membrane, `place` puts stimuli and synapses at sites. Both share the ion, channel and integrator machinery and, with the acceleration and computational power of `brainstate`, support vectorized simulation, automatic differentiation and training.

This tutorial covers the HH framework and point neurons.

## The Hodgkin–Huxley framework

### Electrophysiological Properties

In the Hodgkin–Huxley framework, the membrane potential of a neuron is determined by multiple currents:
- Active currents: Controlled by sodium and potassium channels with voltage-gated properties.
- Passive currents: Such as leak currents.
- Capacitive currents: Arising from changes in membrane potential.

This modeling approach was first proposed by Hodgkin and Huxley in 1952 to explain the mechanism of action potentials in the squid giant axon.
The HH model not only laid the foundation of modern neuronal modeling but also remains a core component of many models today.

The equivalent circuit diagram of the classical HH model is shown below:

![Equivalent circuit diagram](../_static/hh.png)

In the classical HH model, the membrane potential $V$ is governed by the following equation:

$$
C_m \frac{dV}{dt} = - (I_{\text{Na}} + I_{\text{K}} + I_L) + I_{\text{ext}}
$$

Where:
- $C_m$: Membrane capacitance
- $I_{\text{Na}}, I_{\text{K}}$: Ionic currents from sodium and potassium channels
- $I_L$: Leak current
- $I_{\text{ext}}$: External injected current (e.g., current applied through a stimulating electrode)

### Modeling Implementation

The HH machinery lives in {class}`braincell.HHTypedNeuron`, the base class of `Cell`. It inherits from:
- `Dynamics`: For defining state variables and differential equations.
- `Container`: For managing modules.
- `DiffEqModule`: A differential equation module that supports automatic modeling.

You rarely use `HHTypedNeuron` directly: a `Cell` builds the ion and channel objects from its declarations. It is also the `root_type` of channels that need no ion, such as the leak `IL`.

## Point neurons

### Electrophysiological Properties


A point neuron has no explicit spatial structure; it models a conductance-based neuron with a single compartment, used for modeling conductance-based neurons with a single compartment. Its membrane potential is governed by the following differential equation:

$$
C_m \frac{dV}{dt} = \sum_j g_j(E_j - V) + I_{\text{ext}}
$$

Where:
- $C_m$: Membrane capacitance
- $V$: Membrane potential
- $g_j$: Conductance of the $j$-th ion channel
- $E_j$: Reversal potential of the $j$-th channel
- $I_{\text{ext}}$: External injected current

The conductance $g_j$ of each channel is determined by its gating variables:

$$
g_j = \bar{g}_j \cdot M^x \cdot N^y
$$

Where:
- $\bar{g}_j$: Maximum conductance
- $M$: Activation variable
- $N$: Inactivation variable
- $x, y$: Exponents depending on channel type

The gating variables follow first-order kinetics:

$$
\frac{dx}{dt} = \phi_x \cdot \frac{x_\infty(V) - x}{\tau_x(V)}
$$

Or equivalently:

$$
\frac{dx}{dt} = \phi_x \left( \alpha_x (1 - x) - \beta_x x \right)
$$

Where:
- $x \in \{M, N\}$
- $\phi_x$: Temperature factor
- $x_\infty$: Steady-state value
- $\tau_x$: Time constant
- $\alpha_x$, $\beta_x$: Rate constants

### Modeling Implementation

`braincell.Cell.point` declares a point neuron; `paint` adds the ion species and the channels. Here is the classical HH neuron, with sodium (`Na_HH1952`), potassium (`K_HH1952`) and leak (`IL`) currents:

In [1]:
import brainunit as u
import braincell
from braincell import mech

hh = braincell.Cell.point(1, length=10. * u.um, radius=5. * u.um, solver='rk4')
hh.paint(
    mech.Ion('SodiumFixed', E=50. * u.mV),
    mech.Channel('Na_HH1952'),
    mech.Ion('PotassiumFixed', E=-77. * u.mV),
    mech.Channel('K_HH1952'),
    mech.Channel('IL', E=-54.387 * u.mV, g_max=0.03 * (u.mS / u.cm ** 2)),
)

Cell(root='soma', n_branches=1, n_paint_rules=6, n_place_rules=0, initialized=False)

`length` and `radius` give the cylinder whose membrane area turns the densities ($\bar{g}_j$ in mS/cm², $C_m$ in µF/cm²) into totals; these are the defaults. The membrane capacitance is 1 µF/cm² unless you paint a `mech.CableProperty(membrane_capacitance=...)`.

A point neuron runs like any `Cell`: place a current clamp, record the voltage and run. `place` without a site uses the only one, the middle of the compartment:

In [2]:
hh.place(mech.CurrentClamp(delay=5. * u.ms, durations=40. * u.ms, amplitudes=0.1 * u.nA))
hh.record('v', braincell.observe.state('v'))
result = hh.run(dt=0.025 * u.ms, duration=50. * u.ms)
v = result.samples['v'].values
print(v.shape, f"peak {v.max().to_decimal(u.mV):.1f} mV")

(2000, 1) peak 48.2 mV


`V` has one value per neuron (`hh.V.value.shape == (1,)`), with no compartment axis. To drive the neuron from your own loop, or from `brainpy` projections in a network, call `hh.update(I_ext)` with a current or a current density, as in {doc}`../getting_started/first_steps` and the [Overview](overview.ipynb).

## Next steps

A point neuron treats the neuron as a single isopotential point. To model dendrites, axons, and the soma as a spatially connected cable system, continue to the [Cell](cell.ipynb) tutorial, which paints the same mechanisms onto a `Morphology`.